In [1]:
import sys
from pathlib import Path


import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.explain.compare_features import compare_feature_importance_tables
from src.utils.config import CONFIG

In [2]:
def relpath(path: Path) -> str:
    try:
        return str(path.relative_to(CONFIG.root_dir))
    except Exception:
        return str(path.name)

In [3]:
combined = compare_feature_importance_tables(
    pcos_table_path=CONFIG.tables_dir / "pcos_feature_importances.csv",
    endo_table_path=CONFIG.tables_dir / "endo_feature_importances.csv",
    output_path=CONFIG.tables_dir / "shared_feature_comparison.csv"
)

In [4]:
display(Markdown("### Combined feature comparison"))
display(combined.head(30))

### Combined feature comparison

,feature,dataset,importance,conceptual_group
0,num__Follicle No. (R),pcos,0.132302,other
1,num__Follicle No. (L),pcos,0.069366,other
2,num__hair growth(Y/N),pcos,0.066312,other
3,num__Weight gain(Y/N),pcos,0.045520,bmi_body_composition
4,num__Skin darkening (Y/N),pcos,0.044354,other
5,num__Reg.Exercise(Y/N),pcos,0.043021,other
6,num__Pregnant(Y/N),pcos,0.039208,infertility_fertility
7,num__Cycle(R/I),pcos,0.038193,cycle_menstrual
8,num__Fast food (Y/N),pcos,0.035500,other
9,num__Avg. F size (R) (mm),pcos,0.025937,other


In [5]:
combined.head(30)

,feature,dataset,importance,conceptual_group
0,num__Follicle No. (R),pcos,0.132302,other
1,num__Follicle No. (L),pcos,0.069366,other
2,num__hair growth(Y/N),pcos,0.066312,other
3,num__Weight gain(Y/N),pcos,0.045520,bmi_body_composition
4,num__Skin darkening (Y/N),pcos,0.044354,other
5,num__Reg.Exercise(Y/N),pcos,0.043021,other
6,num__Pregnant(Y/N),pcos,0.039208,infertility_fertility
7,num__Cycle(R/I),pcos,0.038193,cycle_menstrual
8,num__Fast food (Y/N),pcos,0.035500,other
9,num__Avg. F size (R) (mm),pcos,0.025937,other


In [6]:
display(Markdown("### Top features by dataset"))

for dataset_name in ["pcos", "endo"]:
    subset = combined[combined["dataset"] == dataset_name].copy()
    subset = subset.sort_values("importance", ascending=False).head(15).reset_index(drop=True)
    display(Markdown(f"#### {dataset_name.upper()}"))
    display(subset)

### Top features by dataset

#### PCOS

,feature,dataset,importance,conceptual_group
0,num__Follicle No. (R),pcos,0.132302,other
1,num__Follicle No. (L),pcos,0.069366,other
2,num__hair growth(Y/N),pcos,0.066312,other
3,num__Weight gain(Y/N),pcos,0.045520,bmi_body_composition
4,num__Skin darkening (Y/N),pcos,0.044354,other
5,num__Reg.Exercise(Y/N),pcos,0.043021,other
6,num__Pregnant(Y/N),pcos,0.039208,infertility_fertility
7,num__Cycle(R/I),pcos,0.038193,cycle_menstrual
8,num__Fast food (Y/N),pcos,0.035500,other
9,num__Avg. F size (R) (mm),pcos,0.025937,other


#### ENDO

,feature,dataset,importance,conceptual_group
0,num__Hormone_Level_Abnormality,endo,0.529860,hormonal_indicators
1,num__Infertility,endo,0.143491,infertility_fertility
2,num__Menstrual_Irregularity,endo,0.138814,cycle_menstrual
3,num__Chronic_Pain_Level,endo,0.068833,pain
4,num__BMI,endo,0.065315,bmi_body_composition
5,num__Age,endo,0.053687,other


In [7]:
group_summary = (
    combined.groupby(["dataset", "conceptual_group"], as_index=False)["importance"]
    .sum()
    .sort_values(["dataset", "importance"], ascending=[True, False])
    .reset_index(drop=True)
)

display(Markdown("### Importance by conceptual group"))
display(group_summary)

### Importance by conceptual group

,dataset,conceptual_group,importance
0,endo,hormonal_indicators,0.529860
1,endo,infertility_fertility,0.143491
2,endo,cycle_menstrual,0.138814
3,endo,pain,0.068833
4,endo,bmi_body_composition,0.065315
5,endo,other,0.053687
6,pcos,other,0.663249
7,pcos,bmi_body_composition,0.120892
8,pcos,hormonal_indicators,0.092513
9,pcos,infertility_fertility,0.061805


In [8]:
shared_groups = (
    group_summary.pivot(index="conceptual_group", columns="dataset", values="importance")
    .fillna(0)
    .reset_index()
)

display(Markdown("### Shared conceptual groups across datasets"))
display(shared_groups)

### Shared conceptual groups across datasets

dataset,conceptual_group,endo,pcos
0,bmi_body_composition,0.065315,0.120892
1,cycle_menstrual,0.138814,0.061541
2,hormonal_indicators,0.529860,0.092513
3,infertility_fertility,0.143491,0.061805
4,other,0.053687,0.663249
5,pain,0.068833,0.000000


In [9]:
display(Markdown("### Notes"))
print("- The combined table lists feature importances from both datasets in one structure.")
print("- The grouped summary helps compare broader clinical themes, not just raw feature names.")
print("- This is more useful for thesis discussion because PCOS and endometriosis do not share many identical raw variables.")

### Notes

- The combined table lists feature importances from both datasets in one structure.
- The grouped summary helps compare broader clinical themes, not just raw feature names.
- This is more useful for thesis discussion because PCOS and endometriosis do not share many identical raw variables.
